In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import numpy as np
from tqdm import tqdm
from sklearn.linear_model import BayesianRidge
import pandas as pd
import pickle
import os
from glob import glob
import torch

In [3]:
gender_mapper = np.load("/pose-independent-anthropometry/data/gender/CAESAR_GENDER_MAPPER.npz")

In [4]:
use_measurements = ['Ankle Circumference (mm)',
                    'Arm Length (Shoulder to Elbow) (mm)',
                    'Arm Length (Shoulder to Wrist) (mm)',
                    'Arm Length (Spine to Wrist) (mm)',
                    'Chest Circumference (mm)',
                    'Crotch Height (mm)',
                    'Head Circumference (mm)',
                    'Hip Circ Max Height (mm)',
                    'Hip Circumference, Maximum (mm)',
                    'Neck Base Circumference (mm)',
                    'Stature (mm)']

In [5]:
measurement_simple_names =  ["ankle left circumference",
                     "arm length (shoulder to elbow)",
                     "arm right length",
                     "arm length (spine to wrist)",
                     "chest circumference",
                     "crotch height",
                     "head circumference",
                     "Hip circumference max height",
                     "hip circumference",
                     "neck circumference",
                     "height"
                     ]

In [47]:
measurement_map = dict(zip(measurement_simple_names, use_measurements))
measurement_map

{'ankle left circumference': 'Ankle Circumference (mm)',
 'arm length (shoulder to elbow)': 'Arm Length (Shoulder to Elbow) (mm)',
 'arm right length': 'Arm Length (Shoulder to Wrist) (mm)',
 'arm length (spine to wrist)': 'Arm Length (Spine to Wrist) (mm)',
 'chest circumference': 'Chest Circumference (mm)',
 'crotch height': 'Crotch Height (mm)',
 'head circumference': 'Head Circumference (mm)',
 'Hip circumference max height': 'Hip Circ Max Height (mm)',
 'hip circumference': 'Hip Circumference, Maximum (mm)',
 'neck circumference': 'Neck Base Circumference (mm)',
 'height': 'Stature (mm)'}

# TRAIN SMPL2REAL

In [6]:
train_subjects_path = "/pose-independent-anthropometry/data/CAESAR_samples/CAESAR_TSOLI_TRAIN_WITHOUT_BAD.txt"
val_subjects_path = "/pose-independent-anthropometry/data/CAESAR_samples/CAESAR_TSOLI_VAL_WITHOUT_BAD.txt"
test_subjects_path = "/pose-independent-anthropometry/data/CAESAR_samples/CAESAR_TSOLI_TEST_WITHOUT_BAD.txt"


In [7]:
# load train, val and test subjects
with open(train_subjects_path,"r") as f:
    train_subject_names = f.read()
train_subject_names = train_subject_names.split("\n")
print(len(train_subject_names))

with open(val_subjects_path,"r") as f:
    val_subjects_names = f.read()
val_subjects_names = val_subjects_names.split("\n")
print(len(val_subjects_names))

with open(test_subjects_path,"r") as f:
    test_subjects_names = f.read()
test_subjects_names = test_subjects_names.split("\n")
print(len(test_subjects_names))

1478
50
370


In [8]:
train_subject_names = train_subject_names + val_subjects_names
print(len(train_subject_names))

1528


In [9]:
preprocessed_measurements_path = "/data/wear3d_preprocessed/fitted_smpl_2_CAESAR_measurements/anthropometry_2023_10_18_23_32_22_smpl_complex_measurements.json"
all_preprocessed_measurements = json.load(open(preprocessed_measurements_path,"r"))


In [10]:
# TRAIN DATA
X_male = []
y_male = []

X_female = []
y_female = []

for scan_name in tqdm(train_subject_names):
    subject_data = all_preprocessed_measurements[scan_name]
    subject_gender_ind = np.where(gender_mapper["names"] == scan_name)[0].item()
    subject_gender = gender_mapper["genders"][subject_gender_ind]
    
    if subject_gender == "Male":
        X_male.append([subject_data["FIT"][m_name] for m_name in use_measurements])
        y_male.append([subject_data["GT"][m_name] for m_name in use_measurements])
    elif subject_gender == "Female":
        X_female.append([subject_data["FIT"][m_name] for m_name in use_measurements])
        y_female.append([subject_data["GT"][m_name] for m_name in use_measurements])

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1528/1528 [00:01<00:00, 1503.34it/s]


In [11]:
training_datasets = {}

training_datasets["male"] = {}
training_datasets["female"] = {}

training_datasets["male"]["X"] = np.array(X_male)
training_datasets["female"]["X"] = np.array(X_female)

training_datasets["male"]["y"] = np.array(y_male)
training_datasets["female"]["y"] = np.array(y_female)

In [12]:
training_datasets["male"]["y"][0,-1] # scale is cm

182.89999389648438

In [13]:
# TEST DATA
X_male_test = []
y_male_test = []

X_female_test = []
y_female_test = []

for scan_name in tqdm(test_subjects_names):
    if scan_name not in all_preprocessed_measurements:
        print(scan_name," not in data")
        continue
    subject_data = all_preprocessed_measurements[scan_name]
    subject_gender_ind = np.where(gender_mapper["names"] == scan_name)[0].item()
    subject_gender = gender_mapper["genders"][subject_gender_ind]
    
    if subject_gender == "Male":
        X_male_test.append([subject_data["FIT"][m_name] for m_name in use_measurements])
        y_male_test.append([subject_data["GT"][m_name] for m_name in use_measurements])
    elif subject_gender == "Female":
        X_female_test.append([subject_data["FIT"][m_name] for m_name in use_measurements])
        y_female_test.append([subject_data["GT"][m_name] for m_name in use_measurements])

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 370/370 [00:00<00:00, 1497.33it/s]

csr2172a  not in data


In [14]:
testing_datasets = {}

testing_datasets["male"] = {}
testing_datasets["female"] = {}

testing_datasets["male"]["X"] = np.array(X_male_test)
testing_datasets["female"]["X"] = np.array(X_female_test)

testing_datasets["male"]["y"] = np.array(y_male_test)
testing_datasets["female"]["y"] = np.array(y_female_test)

In [15]:
testing_datasets["male"]["y"][0,-1] # scale is cm

176.89999389648438

In [16]:
models = {}
res = {}

for gender in ["male","female"]:
    if gender not in models:
        models[gender] = {}
        res[gender] = []
        
    models[gender] = {}
    for i,m_name in enumerate(use_measurements):
        x_training = training_datasets[gender]["X"]
        y_training = training_datasets[gender]["y"][:,i]
        x_testing = testing_datasets[gender]["X"]
        y_testing = testing_datasets[gender]["y"][:,i]

        model = BayesianRidge()
        reg = model.fit(x_training, y_training)
        y_predict = reg.predict(x_testing)

        models[gender][m_name] = reg
        res[gender].append(np.sum(np.abs(y_testing - y_predict),axis=0) / y_testing.shape[0])

In [17]:
res

{'male': [0.6466915113551042,
  1.3741224302842985,
  1.4276876862081396,
  1.3314816147354933,
  1.5743694415575675,
  1.2137122791560513,
  0.8276297908572295,
  2.420483245433519,
  1.1215033395437926,
  1.2596285030171692,
  0.7152672739053397],
 'female': [0.6671755004819682,
  0.7565424634172364,
  1.1097563600328988,
  1.3963013062083522,
  1.3778062488262743,
  0.9096305727696121,
  0.7871069889698551,
  2.427442791511792,
  1.7282622371959455,
  1.760420396856691,
  0.698118740863367]}

In [18]:
final_res = pd.DataFrame([res["male"],res["female"]],
                         columns=use_measurements,
                         index=["male","female"]).transpose()
final_res.loc['mean'] = final_res.mean()
final_res

,male,female
Ankle Circumference (mm),0.646692,0.667176
Arm Length (Shoulder to Elbow) (mm),1.374122,0.756542
Arm Length (Shoulder to Wrist) (mm),1.427688,1.109756
Arm Length (Spine to Wrist) (mm),1.331482,1.396301
Chest Circumference (mm),1.574369,1.377806
Crotch Height (mm),1.213712,0.909631
Head Circumference (mm),0.827630,0.787107
Hip Circ Max Height (mm),2.420483,2.427443
"Hip Circumference, Maximum (mm)",1.121503,1.728262
Neck Base Circumference (mm),1.259629,1.760420


In [19]:
# !mkdir ../data/smpl2real_models/bayesian_ridge_on_2023_10_18_23_32_22_complex2real_trained_on_TSOLI_TRAIN_AND_VAL_without_bad



In [20]:
root = "/pose-independent-anthropometry/data/smpl2real_models/bayesian_ridge_on_2023_10_18_23_32_22_complex2real_trained_on_TSOLI_TRAIN_AND_VAL_without_bad"

pickle.dump(models["male"], open(os.path.join(root,"male.pkl"),"wb"))
pickle.dump(models["female"], open(os.path.join(root,"female.pkl"),"wb"))


# CREATE BASELINE FOR FRONTAL PARTIAL CAESAR

In [21]:
# import sys
# # sys.path.append("..")
# sys.path.append("../SMPL-Anthropometry")

In [22]:
os.chdir("../SMPL-Anthropometry")

In [23]:
from measure import MeasureBody

In [24]:
partial_fits_path = "/data/wear3d_preprocessed/smpl_fittings/2024_01_05_15_33_49"

In [25]:
partial_fits = glob(os.path.join(partial_fits_path,"*.npz"))

In [83]:
for pf_path in partial_fits:
    scan_name = pf_path.split("/")[-1].split(".npz")[0]
    if scan_name not in all_preprocessed_measurements:
        print(scan_name)

csr4174a
nl_5934a
nl_5427a
nl_5249a
csr2758a
nl_1049a
csr0268a
nl_5394a
nl_1398a
nl_1092a
nl_5419a
nl_5295a
nl_5913a
csr1141a
nl_6314a
csr2105a
nl_5387a
nl_5243a
csr1216a
nl_6697a
nl_1159a
nl_5378a
nl_5434a
csr1089a
nl_5440a
nl_1314a
nl_5369a
nl_5840a
nl_5680a
nl_5672a
nl_7061a
nl_5810a
csr1894a
nl_5493a
nl_5905a
nl_1038a
nl_5399a
nl_1298a
csr2977a
nl_5635a
csr1315a
nl_6637a
nl_5891a
nl_5911a
csr0517a
csr1444a
nl_5826a
nl_5673a
nl_6097a
nl_5214a
nl_5686a
nl_6544a
csr2422a
csr1407a
csr1277a
nl_6848a
nl_5209a
nl_5401a
nl_6273a
csr2339a
nl_5617a
nl_1244a
nl_5598a
csr2332a
nl_6676a
nl_6361a
nl_5380a
nl_6337a
csr2744a
csr1258a
csr2804a
nl_1409a
nl_1229a
nl_1280a
nl_5512a
csr0615a
nl_5645a
csr2336a
csr0489a
nl_5776a
nl_6204a
csr2104a
nl_5517a
nl_4042a
nl_5636a
nl_5805a
nl_6756a
csr2174a
csr1323a
csr0197a
csr2424a
csr2592a
csr1473a
csr2622a
nl_6132a
nl_1413a
nl_1306a
nl_1431a
csr2316a
csr3010a
csr1299a
nl_5875a
nl_1150a
nl_5784a
nl_5862a
nl_1435a
nl_6064a
nl_5922a
csr0565a
nl_5949a
csr1882a
c

In [93]:
acc_pred = {"male":{measurement_map[m_name]:[] for m_name in measurement_simple_names},
            "female":{measurement_map[m_name]:[] for m_name in measurement_simple_names}}
            
acc_gt = {"male":{measurement_map[m_name]:[] for m_name in measurement_simple_names},
          "female":{measurement_map[m_name]:[] for m_name in measurement_simple_names}}

MISSED_EXAMPLES = []

for pf_path in partial_fits:
    scan_name = pf_path.split("/")[-1].split(".npz")[0]
    if scan_name not in test_subjects_names:
        continue
        
    if scan_name not in all_preprocessed_measurements:
        MISSED_EXAMPLES.append(scan_name)
        continue
    
    scan_gender_id = np.where(gender_mapper["names"] == scan_name)[0].item()
    scan_gender = gender_mapper["genders"][scan_gender_id].lower()
    
    scan_data = np.load(pf_path)
    scan_shape = torch.from_numpy(scan_data["shape"])
    
    measurer = MeasureBody("smpl")
    measurer.from_body_model(gender=scan_gender, 
                             shape=scan_shape)
    measurer.measure(measurement_simple_names)
    scan_measurements = [measurer.measurements[m_name] for m_name in measurement_simple_names]
    scan_measurements = np.array(scan_measurements).reshape(1,-1)
    
    for m_name in measurement_simple_names:
        tsoli_m_name = measurement_map[m_name]
          
        pred = models[scan_gender][tsoli_m_name].predict(scan_measurements).item()
        acc_pred[scan_gender][tsoli_m_name].append(pred)
          
        gt = all_preprocessed_measurements[scan_name]["GT"][tsoli_m_name]
        acc_gt[scan_gender][tsoli_m_name].append(gt)

    
#     scan_smpl2real_measurements = [models[scan_gender][measurement_map[m_name]].predict(scan_measurements) 
#                                    for m_name in measurement_simple_names]
    
#     break

In [94]:
MISSED_EXAMPLES

['csr2172a']

In [101]:
final_results = {"male":[],
                 "female":[]}

for m_name in use_measurements:
    for gender in ["male","female"]:
        y_pred = np.array(acc_pred[gender][m_name])
        y_test = np.array(acc_gt[gender][m_name])

        final_results[gender].append(np.sum(np.abs(y_test - y_pred),axis=0) / y_test.shape[0])



In [105]:
y_test.shape

(175,)

In [103]:
final_results = pd.DataFrame([final_results["male"],final_results["female"]],
                         columns=use_measurements,
                         index=["male","female"]).transpose()
final_results.loc['mean'] = final_results.mean()
final_results

,male,female
Ankle Circumference (mm),4.535730,5.681033
Arm Length (Shoulder to Elbow) (mm),14.874572,15.667743
Arm Length (Shoulder to Wrist) (mm),26.974298,27.900702
Arm Length (Spine to Wrist) (mm),62.069046,57.963491
Chest Circumference (mm),8.540191,8.633394
Crotch Height (mm),8.671465,6.632006
Head Circumference (mm),1.670094,5.994122
Hip Circ Max Height (mm),10.374096,7.716438
"Hip Circumference, Maximum (mm)",8.568717,8.935903
Neck Base Circumference (mm),15.178919,5.493875
